# DARF v6 — 100-Run Large-Scale Evaluation

**Goal:** Evaluate our best DARF v6 hair-refine method across:
- **4 reference image sets** (different photos of Hermione / Daenerys)
- **5 pose images** (face-aware dense pose variants)
- **5 prompt styles** (formal, royal/fantasy, modern casual, studio portrait, cinematic outdoor)
- **Total: 100 unique generations** with deterministic seeds

**Best config from v6:** `hair_refine_strength=0.55`, `lora_alpha=1.6`, `up_pad=1.6`

**Resume-safe:** skips runs where the final image + CSV row already exist. Eval-only if image exists but not yet recorded.

In [ ]:
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
# BASE_MODEL = "RunDiffusion/Juggernaut-XL-v9"

In [ ]:
import os
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/melik3kara/LoRA-scale-adaptive-feedback.git {REPO_DIR}
else:
    print("Pulling latest...")
    !cd {REPO_DIR} && git pull

!pip install -q gdown
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)
import glob
if not glob.glob(f"{REPO_DIR}/data/loras/*.safetensors"):
    !gdown --folder "https://drive.google.com/drive/folders/1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5" -O {REPO_DIR}/data/loras/
    import shutil
    for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
        target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
        if f != target:
            shutil.move(f, target)

os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)

!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu
!pip install -q "pydantic<2.10" "typing_extensions>=4.13.0" "mediapipe==0.10.14" "protobuf<5" scipy

os.chdir(REPO_DIR)
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Base model: {BASE_MODEL}")

## Section 1 — Pipeline & Scorers

In [ ]:
from pipeline import load_identities, build_pipeline, generate_two_stage, face_local_refine, hair_local_refine
from scorer import FaceScorer
from pose_scorer import PoseScorer
from evaluator import Evaluator
from PIL import Image
from IPython.display import display
import pipeline as P
import numpy as np
import cv2

identities = load_identities()

# Hair descriptions enriched — same as v6 best run
identities["hermione"]["visual_description"] = (
    "front-facing young woman, long bushy brown curly hair, brunette, "
    "hazel eyes, fair skin, looking directly at the camera, black Hogwarts robe"
)
identities["daenerys"]["visual_description"] = (
    "front-facing woman, long straight platinum silver-white hair, icy blonde, "
    "violet eyes, pale skin, looking directly at the camera, regal blue dress"
)

identities["hermione"]["attention_phrases"] = [
    "long bushy brown curly hair", "brunette hair",
    "Hermione_Granger", "black Hogwarts robe",
]
identities["daenerys"]["attention_phrases"] = [
    "long straight platinum silver-white hair", "icy blonde silver hair",
    "dae woman", "regal blue dress",
]

identities["hermione"]["negative_features"] = (
    "blonde, silver hair, platinum hair, white hair, icy hair, crown, "
    "side profile, turned head"
)
identities["daenerys"]["negative_features"] = (
    "brown hair, brunette, dark hair, bushy curly hair, witch robe, school uniform, "
    "side profile, turned head"
)

P._BASE_NEGATIVE = (
    "blurry, low quality, deformed face, cartoon, illustration, anime, 3d render, "
    "three women, three people, third person, extra person, woman in background, "
    "side profile, profile view, turned head, looking sideways"
)

pipe = build_pipeline(identities, base_model=BASE_MODEL)

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
evaluator   = Evaluator(device="cuda")

names = list(identities.keys())
identity_regions = {names[0]: (0, 0, 512, 1024), names[1]: (512, 0, 1024, 1024)}

def detected_face_count(img):
    img_bgr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
    return len(evaluator.face_app.get(img_bgr))

print("Pipeline and scorers ready.")

## Section 2 — Reference Image Sets

4 sets of reference photos (one per character). Used only for ArcFace evaluation, not for generation.

In [ ]:
# Map reference filenames that actually exist in data/reference_faces/
REF_SETS = [
    {
        "hermione": "data/reference_faces/reference_hermione1.png",
        "daenerys": "data/reference_faces/reference_daenerys1.png",
    },
    {
        "hermione": "data/reference_faces/reference_hermione2.jpg",
        "daenerys": "data/reference_faces/reference_daenerys2.jpg",
    },
    {
        "hermione": "data/reference_faces/reference_hermione3.jpeg",
        "daenerys": "data/reference_faces/reference_daenerys3.jpg",
    },
    {
        "hermione": "data/reference_faces/reference_hermione4.jpeg",
        "daenerys": "data/reference_faces/reference_daenerys4.jpeg",
    },
]

# Validate that all files exist
for i, rs in enumerate(REF_SETS):
    for k, path in rs.items():
        if not os.path.exists(path):
            print(f"WARNING: ref_set {i+1} {k} missing: {path}")
        else:
            print(f"OK: ref_set {i+1} {k} -> {path}")

# Pre-extract ArcFace embeddings for all 4 reference sets
print("\nExtracting reference embeddings...")
REF_EMBED_SETS = []
for i, rs in enumerate(REF_SETS):
    embeds = {}
    for k, path in rs.items():
        img = Image.open(path).convert("RGB")
        emb = evaluator.extract_face_embedding(img)
        if emb is None:
            print(f"WARNING: No face detected in ref_set {i+1} {k}!")
        embeds[k] = emb
    REF_EMBED_SETS.append(embeds)
    print(f"ref_set {i+1}: hermione={embeds['hermione'] is not None}, "
          f"daenerys={embeds['daenerys'] is not None}")

print("\nAll reference embeddings extracted.")

## Section 3 — Pose Images

Generate 5 face-aware dense pose variants if not already present.

In [ ]:
from face_aware_pose import make_face_aware_pose, get_face_aware_target_keypoints

pose_dir = "data/pose_images/eval_poses"
os.makedirs(pose_dir, exist_ok=True)

pose_paths = [f"{pose_dir}/pose_{i+1}.png" for i in range(5)]

def _shift_pose(img_pil, dx=0, dy=0):
    """Translate skeleton pixels by (dx, dy) using numpy roll."""
    arr = np.array(img_pil)
    if dy != 0:
        arr = np.roll(arr, dy, axis=0)
        if dy > 0:
            arr[:dy] = 0
        else:
            arr[dy:] = 0
    if dx != 0:
        arr = np.roll(arr, dx, axis=1)
        if dx > 0:
            arr[:, :dx] = 0
        else:
            arr[:, dx:] = 0
    return Image.fromarray(arr)

# 5 variants: standard dense, standard minimal, shifted left, shifted right, shifted down
POSE_CONFIGS = [
    dict(front_facing=True, dense_face=True,  dx=0,   dy=0),
    dict(front_facing=True, dense_face=False, dx=0,   dy=0),
    dict(front_facing=True, dense_face=True,  dx=-20, dy=0),
    dict(front_facing=True, dense_face=True,  dx=20,  dy=0),
    dict(front_facing=True, dense_face=True,  dx=0,   dy=15),
]

EVAL_POSE_IMAGES = []
for i, cfg in enumerate(POSE_CONFIGS):
    if not os.path.exists(pose_paths[i]):
        dx = cfg.pop("dx")
        dy = cfg.pop("dy")
        base = make_face_aware_pose(1024, 1024, **cfg)
        if dx != 0 or dy != 0:
            base = _shift_pose(base, dx, dy)
        base.save(pose_paths[i])
        print(f"Generated pose_{i+1}.png")
    else:
        print(f"Pose {i+1} already exists.")
    EVAL_POSE_IMAGES.append(Image.open(pose_paths[i]).convert("RGB"))

target_keypoints = get_face_aware_target_keypoints(front_facing=True)

print("\nAll 5 poses ready. Displaying...")
for i, img in enumerate(EVAL_POSE_IMAGES):
    print(f"Pose {i+1}:")
    display(img.resize((256, 256)))

## Section 4 — Prompt Variants

5 styles: formal/academic, royal/fantasy, modern casual, studio portrait, cinematic outdoor.

In [ ]:
# Each variant overrides visual_description per identity
# attention_phrases and LoRA config remain constant
PROMPT_VARIANTS = [
    {
        "name": "formal_academic",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, black Hogwarts school robe, ancient library background"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, regal blue dress, ancient library background"
        ),
    },
    {
        "name": "royal_fantasy",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, elegant ceremonial robes, grand throne room with candlelight"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, commanding royal gown, grand throne room with candlelight"
        ),
    },
    {
        "name": "modern_casual",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, casual blue jeans and white top, modern city street background"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, white linen top and trousers, modern city street background"
        ),
    },
    {
        "name": "studio_portrait",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, professional studio lighting, clean white background, "
            "sharp focus close-up portrait"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair, icy blonde, "
            "violet eyes, professional studio lighting, clean white background, "
            "sharp focus close-up portrait"
        ),
    },
    {
        "name": "cinematic_outdoor",
        "hermione_desc": (
            "front-facing young woman, long bushy brown curly hair, brunette, "
            "hazel eyes, wind softly blowing hair, dramatic sunset cliff overlooking ocean"
        ),
        "daenerys_desc": (
            "front-facing woman, long straight platinum silver-white hair shimmering in sunlight, "
            "icy blonde, violet eyes, dramatic sunset cliff overlooking ocean"
        ),
    },
]

print(f"{len(PROMPT_VARIANTS)} prompt variants defined:")
for i, pv in enumerate(PROMPT_VARIANTS):
    print(f"  {i+1}. {pv['name']}")

## Section 5 — DARF v6 Config

Best configuration from v6 hair-refine experiments (strength=0.55).

In [ ]:
V6_TWO_STAGE_CONFIG = dict(
    layout_lora_scales={"hermione": 0.10, "daenerys": 0.30},
    identity_lora_scales={
        "hermione": {"down": 0.10, "mid": 0.20, "up": 0.45},
        "daenerys": {"down": 0.15, "mid": 0.55, "up": 1.50},
    },
    layout_ctrl_scale=1.00,
    identity_ctrl_scale=0.95,
    refine_strength=0.35,
    refine_steps=28,
    layout_steps=28,
    use_regional_attention=True,
    use_spatial_lora_gate=True,
    spatial_gate_block_floor={"down": 0.20, "mid": 0.05, "up": 0.00},
    use_bg_lock=True,
    bg_lock_ratio=0.40,
    bg_lock_padding=24,
    bg_lock_in_layout=True,
    bg_lock_in_identity=False,
    identity_regions=identity_regions,
)

FACE_REFINE_CONFIG = dict(
    refine_strength=0.40,
    crop_pad_ratio=0.5,
    refine_steps=28,
    feather=24,
)

HAIR_REFINE_CONFIG = dict(
    refine_strength=0.55,   # best from Bölüm 7
    refine_steps=28,
    feather=36,
    horiz_pad=0.6,
    up_pad=1.6,
    down_pad=0.2,
    lora_alpha=1.6,
)

print("DARF v6 config loaded.")
print(f"  hair_refine_strength : {HAIR_REFINE_CONFIG['refine_strength']}")
print(f"  stage2_refine_strength: {V6_TWO_STAGE_CONFIG['refine_strength']}")
print(f"  lora_up_hermione     : {V6_TWO_STAGE_CONFIG['identity_lora_scales']['hermione']['up']}")
print(f"  lora_up_daenerys     : {V6_TWO_STAGE_CONFIG['identity_lora_scales']['daenerys']['up']}")

## Section 6 — Output Setup

Create output directory, initialize CSV with existing run IDs for resume-safety.

In [ ]:
import pandas as pd
import time

OUT_DIR  = "data/results/darf_v6_100run"
CSV_PATH = f"{OUT_DIR}/darf_v6_100run_results.csv"
os.makedirs(OUT_DIR, exist_ok=True)

# Load existing run IDs for resume-safety
if os.path.exists(CSV_PATH):
    df_existing = pd.read_csv(CSV_PATH)
    existing_run_ids = set(df_existing["run_id"].tolist())
    print(f"Resuming: {len(existing_run_ids)}/100 runs already in CSV.")
else:
    existing_run_ids = set()
    print("Starting fresh — no existing CSV found.")

CSV_COLUMNS = [
    "run_id", "ref_idx", "pose_idx", "prompt_idx", "prompt_name",
    "seed", "ref_hermione_path", "ref_daenerys_path", "pose_path",
    "final_image_path",
    "arcface_hermione", "arcface_daenerys", "arcface_mean",
    "face_count", "extra_person_flag", "pose_metric",
    "runtime_seconds",
    "hair_refine_strength", "stage2_refine_strength",
    "lora_up_hermione", "lora_up_daenerys",
]

print(f"Output dir : {OUT_DIR}")
print(f"CSV path   : {CSV_PATH}")
print(f"Total runs : 4 ref × 5 pose × 5 prompt = 100")

## Section 7 — Main 100-Run Loop

Runs all 100 configurations. Resume-safe: skips if image + CSV row exist; eval-only if image exists but not in CSV.

In [ ]:
total = 4 * 5 * 5
done  = len(existing_run_ids)

for ref_idx in range(4):
    for pose_idx in range(5):
        for prompt_idx in range(5):

            run_id = ref_idx * 25 + pose_idx * 5 + prompt_idx
            seed   = 42 + run_id
            pv     = PROMPT_VARIANTS[prompt_idx]
            rs     = REF_SETS[ref_idx]

            final_path = f"{OUT_DIR}/run_{run_id:03d}_ref{ref_idx+1}_pose{pose_idx+1}_p{prompt_idx+1}.png"

            image_exists = os.path.exists(final_path)
            in_csv       = run_id in existing_run_ids

            if image_exists and in_csv:
                continue

            print(f"\n{'='*60}")
            print(f"Run {run_id:03d}/{total-1} | ref={ref_idx+1} pose={pose_idx+1} prompt={prompt_idx+1} ({pv['name']}) | seed={seed}")

            t0 = time.time()

            if image_exists and not in_csv:
                # Eval-only: image already generated, just re-evaluate
                print("  [eval-only mode] image exists, evaluating...")
                final_img = Image.open(final_path).convert("RGB")
            else:
                # Full pipeline
                identities["hermione"]["visual_description"] = pv["hermione_desc"]
                identities["daenerys"]["visual_description"] = pv["daenerys_desc"]

                pose_img = EVAL_POSE_IMAGES[pose_idx]

                _, stage2 = generate_two_stage(
                    pipe, identities, pose_img, seed=seed, **V6_TWO_STAGE_CONFIG
                )
                after_face = face_local_refine(
                    pipe, identities, stage2, face_scorer, identity_regions,
                    seed=seed, **FACE_REFINE_CONFIG
                )
                final_img = hair_local_refine(
                    pipe, identities, after_face, face_scorer, identity_regions,
                    seed=seed, **HAIR_REFINE_CONFIG
                )
                final_img.save(final_path)

            runtime = time.time() - t0

            # Evaluate: ArcFace
            ref_embeds = REF_EMBED_SETS[ref_idx]
            sim_h, sim_d = evaluator.detect_and_assign_faces(
                final_img, ref_embeds["hermione"], ref_embeds["daenerys"]
            )
            n_faces = detected_face_count(final_img)

            # Evaluate: Pose metric
            try:
                pose_err = evaluator.pose_keypoint_error(final_img, EVAL_POSE_IMAGES[pose_idx])
            except Exception as ex:
                print(f"  [pose eval error]: {ex}")
                pose_err = -1.0

            row = {
                "run_id":               run_id,
                "ref_idx":              ref_idx + 1,
                "pose_idx":             pose_idx + 1,
                "prompt_idx":           prompt_idx + 1,
                "prompt_name":          pv["name"],
                "seed":                 seed,
                "ref_hermione_path":    rs["hermione"],
                "ref_daenerys_path":    rs["daenerys"],
                "pose_path":            pose_paths[pose_idx],
                "final_image_path":     final_path,
                "arcface_hermione":     round(float(sim_h), 4),
                "arcface_daenerys":     round(float(sim_d), 4),
                "arcface_mean":         round((float(sim_h) + float(sim_d)) / 2, 4),
                "face_count":           n_faces,
                "extra_person_flag":    int(n_faces > 2),
                "pose_metric":          round(float(pose_err), 4),
                "runtime_seconds":      round(runtime, 1),
                "hair_refine_strength": HAIR_REFINE_CONFIG["refine_strength"],
                "stage2_refine_strength": V6_TWO_STAGE_CONFIG["refine_strength"],
                "lora_up_hermione":     V6_TWO_STAGE_CONFIG["identity_lora_scales"]["hermione"]["up"],
                "lora_up_daenerys":     V6_TWO_STAGE_CONFIG["identity_lora_scales"]["daenerys"]["up"],
            }

            existing_run_ids.add(run_id)
            done += 1

            write_header = not os.path.exists(CSV_PATH)
            pd.DataFrame([row]).to_csv(CSV_PATH, mode="a", header=write_header, index=False)

            print(f"  arc_h={sim_h:.3f}  arc_d={sim_d:.3f}  mean={row['arcface_mean']:.3f}  "
                  f"faces={n_faces}  extra={row['extra_person_flag']}  "
                  f"pose_err={pose_err:.3f}  {runtime:.0f}s  [{done}/{total} done]")

            # Show image every 10 runs
            if done % 10 == 0:
                display(final_img.resize((384, 384)))

print(f"\n{'='*60}")
print(f"Loop complete. {done}/{total} runs done. CSV at {CSV_PATH}")

## Section 8 — Results Analysis

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")
df.head(10)

In [ ]:
# Overall statistics
numeric_cols = ["arcface_hermione", "arcface_daenerys", "arcface_mean",
                "face_count", "extra_person_flag", "pose_metric", "runtime_seconds"]

stats = df[numeric_cols].describe().T
stats["std"] = df[numeric_cols].std()

print("=== Overall Statistics ===")
print(stats[["mean", "std", "min", "50%", "max"]].round(4))

print(f"\n=== Summary ===")
print(f"Total runs            : {len(df)}")
print(f"ArcFace mean ± std    : {df['arcface_mean'].mean():.4f} ± {df['arcface_mean'].std():.4f}")
print(f"ArcFace Hermione mean : {df['arcface_hermione'].mean():.4f} ± {df['arcface_hermione'].std():.4f}")
print(f"ArcFace Daenerys mean : {df['arcface_daenerys'].mean():.4f} ± {df['arcface_daenerys'].std():.4f}")
print(f"Both identity success (arc > 0.25 each): "
      f"{((df['arcface_hermione'] > 0.25) & (df['arcface_daenerys'] > 0.25)).sum()} / {len(df)}")
print(f"Extra person rate     : {df['extra_person_flag'].mean()*100:.1f}%")
print(f"Exactly 2 faces rate  : {(df['face_count'] == 2).mean()*100:.1f}%")
print(f"Mean runtime          : {df['runtime_seconds'].mean():.0f}s  Total: {df['runtime_seconds'].sum()/3600:.1f}h")

In [ ]:
# Group by prompt style
print("=== By Prompt Style ===")
grp_prompt = df.groupby("prompt_name").agg(
    arc_mean=("arcface_mean", "mean"),
    arc_h_mean=("arcface_hermione", "mean"),
    arc_d_mean=("arcface_daenerys", "mean"),
    extra_person_rate=("extra_person_flag", "mean"),
    pose_err=("pose_metric", "mean"),
    n=("run_id", "count"),
).round(4)
print(grp_prompt.to_string())

In [ ]:
# Group by pose
print("=== By Pose Index ===")
grp_pose = df.groupby("pose_idx").agg(
    arc_mean=("arcface_mean", "mean"),
    arc_h=("arcface_hermione", "mean"),
    arc_d=("arcface_daenerys", "mean"),
    extra_person_rate=("extra_person_flag", "mean"),
    pose_err=("pose_metric", "mean"),
    n=("run_id", "count"),
).round(4)
print(grp_pose.to_string())

In [ ]:
# Group by reference set
print("=== By Reference Set ===")
grp_ref = df.groupby("ref_idx").agg(
    arc_mean=("arcface_mean", "mean"),
    arc_h=("arcface_hermione", "mean"),
    arc_d=("arcface_daenerys", "mean"),
    extra_person_rate=("extra_person_flag", "mean"),
    n=("run_id", "count"),
).round(4)
print(grp_ref.to_string())

In [ ]:
# Top 5 best runs by arcface_mean
print("=== Top 5 Runs (arcface_mean) ===")
top5 = df.nlargest(5, "arcface_mean")[["run_id", "ref_idx", "pose_idx", "prompt_name",
                                        "arcface_hermione", "arcface_daenerys", "arcface_mean",
                                        "face_count", "final_image_path"]]
print(top5.to_string(index=False))

print("\n=== Bottom 5 Runs (arcface_mean) ===")
bot5 = df.nsmallest(5, "arcface_mean")[["run_id", "ref_idx", "pose_idx", "prompt_name",
                                         "arcface_hermione", "arcface_daenerys", "arcface_mean",
                                         "face_count", "final_image_path"]]
print(bot5.to_string(index=False))

In [ ]:
# Show top 5 best images
print("Top 5 best-scoring images:")
for _, row in top5.iterrows():
    path = row["final_image_path"]
    if os.path.exists(path):
        img = Image.open(path)
        print(f"run_{int(row['run_id']):03d} | {row['prompt_name']} | "
              f"arc_h={row['arcface_hermione']:.3f} arc_d={row['arcface_daenerys']:.3f}")
        display(img.resize((384, 384)))

## Section 9 — Save Summary CSVs + ZIP

In [ ]:
# Save group summary CSVs
grp_prompt.to_csv(f"{OUT_DIR}/summary_by_prompt.csv")
grp_pose.to_csv(f"{OUT_DIR}/summary_by_pose.csv")
grp_ref.to_csv(f"{OUT_DIR}/summary_by_ref.csv")

# Full results already at CSV_PATH
print(f"Saved summary CSVs to {OUT_DIR}/")

# Create ZIP
import shutil
from IPython.display import FileLink

zip_path = "/workspace/darf_v6_100run_results"
shutil.make_archive(zip_path, "zip", OUT_DIR)
print(f"ZIP ready: {zip_path}.zip")
FileLink(f"{zip_path}.zip")